# CSE 151B Competition — Starter Notebook

Welcome to the **CSE 151B Spring 2026 Math Reasoning Competition**!  
This notebook walks you through the full pipeline end-to-end:

1. Setting up the Python environment with `uv`
2. Loading the competition dataset
3. Running inference with **Qwen3-4B-Thinking** via vLLM (INT8 quantized)
4. Scoring responses against ground-truth answers
5. Saving results to JSONL for submission

The public dataset (`public.jsonl`) contains questions **with** answers so you can measure accuracy locally.  
The private test set used for the leaderboard does **not** include answers — for that, skip evaluation and submit the raw responses.

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### Comment Out the cell below after first installation.

In [1]:
# # Install uv
# !wget -qO- https://astral.sh/uv/install.sh | sh

# # Create a virtual environment using Python 3.12 explicitly
# !~/.local/bin/uv venv .venv --seed --python 3.12

# # Install dependencies — this is fast thanks to uv's parallel resolver
# !.venv/bin/python -m pip install sympy numpy transformers vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# # Install Jupyter Kernel
# !.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"
# print("Done. Restart the kernel before proceeding.")
# print("Selection process: on top right, click on current kernel '(usually named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

### Run the cell below every time to activate the installed environment. 

In [2]:
# # activate venv after installation. This needs to be run everytime.
# !source ./.venv/bin/activate

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [3]:
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
DATA_PATH   = "data/public.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"
MAX_TOKENS  = 32768

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 4. Prompt Construction

We use two system prompts depending on the question type:

- **MCQ** — the model must select the best answer letter and wrap it in `\boxed{}`
- **Free-form** — the model solves step-by-step and puts the final answer in `\boxed{}`

`build_prompt()` returns the appropriate `(system, user)` pair for each item.

In [5]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question


# Verify with samples
for label, item in [("MCQ", mcq_sample), ("Free-form", free_sample)]:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 200 chars) ──")
    print(usr_p[:200], "...\n")

── MCQ user prompt (first 200 chars) ──
$int_{-infty}^{+infty} frac{a^{3/2}}{s^2+a^2} ds = $

Options:
A. $0$
B. $frac{1}{a}$
C. $frac{3}{a}$
D. $frac{1}{2a^2}$
E. $frac{1}{2a}$
F. $frac{2}{a}$
G. $2a$
H. $frac{3}{2a}$
I. $frac{3}{2a^2}$
J. ...

── Free-form user prompt (first 200 chars) ──
Find the sum of the first $325$ positive even whole numbers. Sum: [ANS] ...



## 5.5 Supervised Fine Tuning

In [ ]:
# Load sft dataset
from datasets import load_dataset
from peft import LoraConfig, get_peft_model

sft_data = load_dataset("AI-MO/NuminaMath-CoT")

In [ ]:
# ── Step 1: Filter to synthetic_math and orca_math subsets ──

train_dataset = sft_data["train"].filter(
    lambda x: x["source"] in ["synthetic_math", "orca_math"]
)
print(f"Filtered dataset size: {len(train_dataset)} examples")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token


In [ ]:
# ── Step 2: Format examples with </think> tag ──

def format_example(example):
    """
    Format an example for SFT training.
    Inserts </think> tag before the line containing \boxed{}.
    """
    solution = example["solution"]
    
    # Find the first line containing \boxed{
    lines = solution.split("\n")
    boxed_line_idx = None
    for i, line in enumerate(lines):
        if "\\boxed{" in line:
            boxed_line_idx = i
            break
    
    # If no \boxed{ found, use the solution as-is
    if boxed_line_idx is None:
        modified_solution = solution
    else:
        # Insert <\think> tag on its own line before the boxed line
        lines_before = lines[:boxed_line_idx]
        lines_after = lines[boxed_line_idx:]
        modified_solution = "\\n".join(lines_before) + "\\n\\n</think>\\n\\n" + "\\n".join(lines_after)
    
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": example["problem"]}],
        tokenize=False,
        add_generation_prompt=True,
    )
    
    return {
        "prompt": prompt,
        "completion": modified_solution,
    }

formatted_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
print("First example completion (first 500 chars):")
print(formatted_dataset[0]["completion"][:500])


In [ ]:
# ── Step 3a: Load base model in 4-bit QLoRA ──

from transformers import AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
base_model.config.use_cache = False
base_model.enable_input_require_grads()
print("Base model loaded in 4-bit.")


In [ ]:
# ── Step 3b: Attach LoRA adapters ──

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
    bias="none",
)

peft_model = get_peft_model(base_model, lora_config)
print("LoRA adapters attached.")


In [ ]:
# ── Step 3c: Add "text" column and set up training ──

def create_text_column(example):
    """Concatenate prompt and completion into a single 'text' column."""
    return {"text": example["prompt"] + example["completion"]}

formatted_dataset = formatted_dataset.map(create_text_column)

training_args = TrainingArguments(
    output_dir="checkpoints/sft_qwen3_4b",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    optim="paged_adamw_8bit",
    report_to="none",
    gradient_checkpointing=True,
)

print("Training arguments configured.")


In [ ]:
# ── Step 3d: Create and run SFTTrainer ──

from trl import SFTTrainer

trainer = SFTTrainer(
    model=peft_model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,
    dataset_text_field="text",
    max_seq_length=4096,
    args=training_args,
)

trainer.train()
print("Training complete.")


In [ ]:
# ── Step 4: Save the fine-tuned model ──

merged_model = peft_model.merge_and_unload()
MERGED_MODEL_PATH = "checkpoints/sft_qwen3_4b/merged"
merged_model.save_pretrained(MERGED_MODEL_PATH)
tokenizer.save_pretrained(MERGED_MODEL_PATH)
print(f"Merged model and tokenizer saved to {MERGED_MODEL_PATH}")


In [ ]:
INFERENCE_MODEL = "checkpoints/sft_qwen3_4b/merged"

tokenizer = AutoTokenizer.from_pretrained(INFERENCE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=INFERENCE_MODEL,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=16384,
    trust_remote_code=True,
    max_num_seqs=256,
    max_num_batched_tokens=32768,
)

sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
)

print("Model loaded.")